# SIGMA 2.0: Training Model Hybrid (MobileSAM + LSTM) di Google Colab
Jalankan notebook ini di Google Colab dengan memastikan Runtime menggunakan **GPU** (Menu: Runtime -> Change runtime type -> T4 GPU).

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Persiapan Environment & Download Weights MobileSAM

In [ ]:
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git
!pip install -q timm tqdm opencv-python

# Download pre-trained weights MobileSAM ke /content
!wget -q https://raw.githubusercontent.com/ChaoningZhang/MobileSAM/master/weights/mobile_sam.pt -O /content/mobile_sam.pt
print("Instalasi library dan download bobot MobileSAM selesai!")

### 3. Ekstrak Dataset dari Google Drive
Pastikan Anda telah mengompres folder `videos` yang berisi kelas gerakan (misal: `A`, `Beli`, dll) menjadi `dataset_bisindo.zip` dan mengunggahnya ke Google Drive utama Anda.

In [ ]:
import os
import zipfile

# Sesuaikan path ini dengan lokasi file ZIP Anda di Google Drive
zip_path = '/content/drive/MyDrive/dataset_bisindo.zip'
extract_path = '/content/dataset'

if os.path.exists(zip_path):
    print(f"Mengekstrak {zip_path} ke local storage Colab...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Ekstraksi selesai! Data siap digunakan.")
else:
    print(f"ERROR: File {zip_path} tidak ditemukan! Pastikan nama dan lokasi file di Drive sudah benar.")

### 4. Definisi Kelas & Arsitektur Model

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time
from tqdm import tqdm
from mobile_sam import sam_model_registry, SamPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan Device: {device}")

class BISINDOFeatureDataset(Dataset):
    def __init__(self, features_dir, classes):
        self.features_dir = features_dir
        self.classes = classes
        self.samples = []
        for class_idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(features_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for feat_file in os.listdir(class_dir):
                if feat_file.endswith('.pt'):
                    self.samples.append((os.path.join(class_dir, feat_file), class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feat_path, class_idx = self.samples[idx]
        features = torch.load(feat_path)
        return features, class_idx

class SignLanguageLSTM(nn.Module):
    def __init__(self, input_size=1024, hidden_size=256, num_layers=2, num_classes=76):
        super(SignLanguageLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

### 5. Ekstraksi Fitur MobileSAM & Training LSTM

In [ ]:
def extract_and_cache_features(data_dir, features_dir, sam_checkpoint, classes, frames_per_seq=45):
    print("Memuat model MobileSAM untuk Feature Extraction...")
    sam = sam_model_registry["vit_t"](checkpoint=sam_checkpoint)
    sam.to(device=device)
    sam.eval()
    predictor = SamPredictor(sam)
    pool = nn.AdaptiveAvgPool2d((2, 2))
    
    os.makedirs(features_dir, exist_ok=True)
    for class_name in classes:
        class_data_dir = os.path.join(data_dir, class_name)
        class_feat_dir = os.path.join(features_dir, class_name)
        if not os.path.isdir(class_data_dir): continue
        os.makedirs(class_feat_dir, exist_ok=True)
        
        sequences = os.listdir(class_data_dir)
        for seq_folder in tqdm(sequences, desc=f"Ekstraksi {class_name}"):
            seq_path = os.path.join(class_data_dir, seq_folder)
            feat_path = os.path.join(class_feat_dir, f"{seq_folder}.pt")
            feat_flip_path = os.path.join(class_feat_dir, f"{seq_folder}_flip.pt")
            
            if os.path.exists(feat_path) and os.path.exists(feat_flip_path): continue
            if not os.path.isdir(seq_path): continue
            
            frame_files = sorted([f for f in os.listdir(seq_path) if f.endswith('.jpg')])
            features_normal, features_flipped = [], []
            
            with torch.no_grad():
                for i in range(frames_per_seq):
                    if i < len(frame_files):
                        img_path = os.path.join(seq_path, frame_files[i])
                        img = cv2.imread(img_path)
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    
                    # Normal
                    predictor.set_image(img)
                    features_normal.append(pool(predictor.features).view(-1).cpu())
                    
                    # Flipped (Augmentasi Tangan Kiri)
                    img_flip = cv2.flip(img, 1)
                    predictor.set_image(img_flip)
                    features_flipped.append(pool(predictor.features).view(-1).cpu())
            
            torch.save(torch.stack(features_normal), feat_path)
            torch.save(torch.stack(features_flipped), feat_flip_path)

print("=== TAHAP 1: EKSTRAKSI FITUR MOBILESAM ===")
# Sesuaikan jika path hasil ekstrak Anda berbeda (misal: /content/dataset/videos)
data_dir = '/content/dataset/videos'
features_dir = '/content/features_cache'
sam_checkpoint = '/content/mobile_sam.pt'

classes = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", 
           "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z",
           "Beli", "Jual", "Bawa", "Lihat", "Ambil", "Buang", "Cari",
           "Tas", "Mobil", "Pensil", "Baju", "Buku", "Meja", "Kursi", "Lemari",
           "Mahal", "Murah", "Bagus", "Jelek", "Besar", "Kecil", "Panjang", "Tinggi", "Berat",
           "Sekarang", "Kemarin", "Sering", "Jarang", "Tidak",
           "Aku", "Kamu", "Kita", "Kami", "Mereka", "Ini", "Itu",
           "Atas", "Bawah", "Depan", "Belakang", "Luar", "Samping",
           "Halo", "Iya", "Maaf", "Tolong", "Terima kasih", "Sama-sama",
           "Apa", "Kenapa"]

extract_and_cache_features(data_dir, features_dir, sam_checkpoint, classes, frames_per_seq=45)

print("\n=== TAHAP 2: TRAINING MODEL LSTM ===")
# Simpan model langsung ke Google Drive agar tidak hilang
models_dir = '/content/drive/MyDrive/SIGMA2_Models'
os.makedirs(models_dir, exist_ok=True)

dataset = BISINDOFeatureDataset(features_dir, classes)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = SignLanguageLSTM(input_size=1024, hidden_size=256, num_layers=2, num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for features, labels in dataloader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {total_loss/len(dataloader):.4f} | Accuracy: {acc:.2f}%")
    
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), os.path.join(models_dir, f'lstm_epoch_{epoch+1}.pth'))

final_model_path = os.path.join(models_dir, 'lstm_final.pth')
torch.save(model.state_dict(), final_model_path)
print(f"\nTraining selesai! Model final tersimpan di Google Drive: {final_model_path}")